### Train the dataset

In [3]:
import os
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import RandomizedSearchCV, KFold, ParameterSampler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint, uniform, loguniform

warnings.filterwarnings('ignore')

# ── GPU / CPU AUTO-DETECT ─────────────────────────────────────────────────────
USE_GPU_RF  = False
USE_GPU_XGB = False

try:
    from cuml.ensemble import RandomForestRegressor
    USE_GPU_RF = True
    print("[GPU] cuML RandomForestRegressor  ✓")
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    print("[CPU] sklearn RandomForestRegressor  ✓")

try:
    from xgboost import XGBRegressor
    import subprocess, sys
    test = subprocess.run(
        [sys.executable, '-c',
         'import xgboost as x; x.XGBRegressor(device="cuda").fit([[1]],[1])'],
        capture_output=True, timeout=10
    )
    USE_GPU_XGB = test.returncode == 0
    print(f"[{'GPU' if USE_GPU_XGB else 'CPU'}] XGBoost "
          f"({'CUDA' if USE_GPU_XGB else 'CPU fallback'})  ✓")
except Exception:
    from xgboost import XGBRegressor
    print("[CPU] XGBoost  ✓")
print()

# ── CONFIG ────────────────────────────────────────────────────────────────────
PRODUCTION_FILE = '../data/coffee_production_bener_meriah.csv'
CLIMATE_FILE    = '../data/climateData_BenerMeriah_2020-01-01_2024-12-31.csv'
RESULT_DIR      = '../results'
RANDOM_STATE    = 42
CV_FOLDS        = 5
N_ITER_SEARCH   = 100

os.makedirs(RESULT_DIR, exist_ok=True)

# ── STEP 1 — Production data ──────────────────────────────────────────────────
print("=" * 65)
print("STEP 1 — Loading production data")
print("=" * 65)

prod_df = pd.read_csv(PRODUCTION_FILE)
prod_df['village']     = prod_df['village'].str.strip().str.lower()
prod_df['tm_ha']       = pd.to_numeric(prod_df['tm_ha'],       errors='coerce')
prod_df['produksi_kg'] = pd.to_numeric(prod_df['produksi_kg'], errors='coerce')
prod_df = prod_df[prod_df['tm_ha'] > 0].dropna(subset=['tm_ha', 'produksi_kg']).copy()
prod_df['yield_kg_ha'] = prod_df['produksi_kg'] / prod_df['tm_ha']
prod_df = prod_df.dropna(subset=['yield_kg_ha']).reset_index(drop=True)

print(f"  Production years : {sorted(prod_df['year'].unique())}")
print(f"  Villages         : {prod_df['village'].nunique()}")
print(f"  Total records    : {len(prod_df)}")
print(f"  yield_kg_ha      : min={prod_df['yield_kg_ha'].min():.2f}  "
      f"mean={prod_df['yield_kg_ha'].mean():.2f}  "
      f"max={prod_df['yield_kg_ha'].max():.2f}\n")

# ── STEP 2 — Climate data ─────────────────────────────────────────────────────
print("=" * 65)
print("STEP 2 — Loading & aggregating climate data")
print("=" * 65)

climate_df = pd.read_csv(CLIMATE_FILE, parse_dates=['date'])
climate_df['year']     = climate_df['date'].dt.year
climate_df['location'] = climate_df['location'].str.strip().str.lower()

CLIMATE_VARS = [c for c in [
    'rainfall_mm', 'temperature_celsius', 'relative_humidity_percent',
    'soil_moisture_percent', 'wind_speed_10m', 'dtr_celsius',
    'vpd_kpa', 'net_solar_rad_kwh_m2',
    'solar_radiation_kwh_m2', 'soil_moisture_layer2_pct',
    'soil_moisture_layer3_pct', 'soil_temp_celsius',
    'tmax_celsius', 'tmin_celsius',
] if c in climate_df.columns]

climate_agg = (
    climate_df
    .groupby(['year', 'location'])[CLIMATE_VARS]
    .mean()
    .reset_index()
)
climate_agg['prod_year'] = climate_agg['year'] + 1

print(f"  Climate years : {sorted(climate_agg['year'].unique())}")
print(f"  Locations     : {climate_agg['location'].nunique()}")
print(f"  Features      : {CLIMATE_VARS}\n")

# ── STEP 3 — Lag merge ───────────────────────────────────────────────────────
print("=" * 65)
print("STEP 3 — Lag merge  (climate year N → production year N+1)")
print("=" * 65)

merged = pd.merge(
    climate_agg,
    prod_df[['year', 'village', 'tm_ha', 'produksi_kg', 'yield_kg_ha']],
    left_on  = ['prod_year', 'location'],
    right_on = ['year',      'village'],
    how      = 'inner'
).drop(columns=['year_y']).rename(columns={'year_x': 'climate_year'})

print(f"  Merged records   : {len(merged)}")
print(f"  Villages matched : {merged['village'].nunique()}")
print(f"  Production years : {sorted(merged['prod_year'].unique())}\n")

# ── STEP 4 — Split ───────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 4 — Train / Validate split")
print("=" * 65)

train_df    = merged[merged['prod_year'] <= 2024].copy()
validate_df = merged[merged['prod_year'] == 2025].copy()
has_val     = len(validate_df) > 0

print(f"  Training   : {len(train_df)} records  "
      f"| prod years {sorted(train_df['prod_year'].unique())}")
if has_val:
    print(f"  Validation : {len(validate_df)} records  | prod year 2025")
else:
    print("  Validation : 2025 not available — skipped")
print()

FEATURES = CLIMATE_VARS
X_train  = train_df[FEATURES].values
y_train  = train_df['yield_kg_ha'].values
if has_val:
    X_val = validate_df[FEATURES].values
    y_val = validate_df['yield_kg_ha'].values

# ── HYPERPARAMETER SPACES ────────────────────────────────────────────────────
rf_param_dist = {
    'n_estimators'             : randint(50, 1000),
    'max_depth'                : [None, 3, 5, 8, 10, 15, 20, 25, 30, 40],
    'min_samples_split'        : randint(2, 30),
    'min_samples_leaf'         : randint(1, 20),
    'max_features'             : ['sqrt', 'log2', 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0],
    'bootstrap'                : [True, False],
    'max_leaf_nodes'           : [None, 10, 20, 50, 100, 200, 500],
    'min_weight_fraction_leaf' : uniform(0.0, 0.1),
    'min_impurity_decrease'    : uniform(0.0, 0.05),
}
if USE_GPU_RF:
    rf_param_dist = {
        'n_estimators'      : randint(50, 1000),
        'max_depth'         : [3, 5, 8, 10, 15, 20, 25, 30],
        'min_samples_split' : randint(2, 30),
        'min_samples_leaf'  : randint(1, 20),
        'max_features'      : ['sqrt', 'log2', 0.3, 0.5, 0.7, 1.0],
        'max_leaves'        : [0, 8, 16, 32, 64, 128],
    }

xgb_param_dist = {
    'n_estimators'      : randint(50, 1000),
    'learning_rate'     : loguniform(1e-3, 0.5),
    'max_depth'         : randint(2, 15),
    'min_child_weight'  : randint(1, 20),
    'subsample'         : uniform(0.4, 0.6),
    'colsample_bytree'  : uniform(0.4, 0.6),
    'colsample_bylevel' : uniform(0.4, 0.6),
    'colsample_bynode'  : uniform(0.4, 0.6),
    'gamma'             : uniform(0.0, 2.0),
    'reg_alpha'         : loguniform(1e-4, 10.0),
    'reg_lambda'        : loguniform(1e-4, 10.0),
    'max_delta_step'    : randint(0, 10),
    'scale_pos_weight'  : uniform(0.5, 2.0),
    'max_bin'           : [64, 128, 256, 512],
    'grow_policy'       : ['depthwise', 'lossguide'],
    'max_leaves'        : randint(0, 64),
}

# ── HELPERS ──────────────────────────────────────────────────────────────────
def tune_and_train(model, param_dist, X, y, name):
    print(f"  Tuning {name}  |  {N_ITER_SEARCH} iterations  |  {CV_FOLDS}-fold CV ...")
    cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    search = RandomizedSearchCV(
        estimator           = model,
        param_distributions = param_dist,   # pass directly, no pre-sampling
        n_iter              = N_ITER_SEARCH,
        scoring             = 'neg_root_mean_squared_error',
        cv                  = cv,
        random_state        = RANDOM_STATE,
        n_jobs              = -1,
        verbose             = 1,
        error_score         = 'raise',
        refit               = True,
    )
    search.fit(X, y)
    cv_rmse = -search.best_score_
    print(f"\n  Best CV RMSE : {cv_rmse:.4f} kg/ha")
    print(f"  Best params  :")
    for k, v in search.best_params_.items():
        print(f"    {k:30s} = {v}")
    pd.DataFrame(search.cv_results_).sort_values('rank_test_score').to_csv(
        os.path.join(RESULT_DIR,
                     f'{name.lower().replace(" ", "_")}_cv_results.csv'),
        index=False)
    print(f"  Full CV results saved.\n")
    return search.best_estimator_, search.best_params_, cv_rmse

def evaluate(model, X, y, label):
    preds = model.predict(X)
    rmse  = np.sqrt(mean_squared_error(y, preds))
    mae   = mean_absolute_error(y, preds)
    r2    = r2_score(y, preds)
    mape  = np.mean(np.abs((y - preds) / np.where(y == 0, 1, y))) * 100
    print(f"    [{label}]  RMSE={rmse:.2f}  MAE={mae:.2f}  "
          f"R²={r2:.4f}  MAPE={mape:.2f}%")
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape, 'predictions': preds}

# ── STEP 5 — Random Forest ───────────────────────────────────────────────────
print("=" * 65)
print(f"STEP 5 — Random Forest  [{'GPU — cuML' if USE_GPU_RF else 'CPU — sklearn'}]")
print("=" * 65)
rf_model, rf_best_params, rf_cv_rmse = tune_and_train(
    RandomForestRegressor(random_state=RANDOM_STATE),
    rf_param_dist, X_train, y_train, "Random Forest")
print("  Performance:")
rf_train_m = evaluate(rf_model, X_train, y_train, "Train")
if has_val:
    rf_val_m = evaluate(rf_model, X_val, y_val, "Validate")
print()

# ── STEP 6 — XGBoost ─────────────────────────────────────────────────────────
print("=" * 65)
print(f"STEP 6 — XGBoost  [{'GPU — CUDA' if USE_GPU_XGB else 'CPU'}]")
print("=" * 65)
xgb_model, xgb_best_params, xgb_cv_rmse = tune_and_train(
    XGBRegressor(random_state=RANDOM_STATE, verbosity=0,
                 objective='reg:squarederror', tree_method='hist',
                 device='cuda' if USE_GPU_XGB else 'cpu'),
    xgb_param_dist, X_train, y_train, "XGBoost")
print("  Performance:")
xgb_train_m = evaluate(xgb_model, X_train, y_train, "Train")
if has_val:
    xgb_val_m = evaluate(xgb_model, X_val, y_val, "Validate")
print()

# ── STEP 7 — Save ────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 7 — Saving models")
print("=" * 65)
joblib.dump(rf_model,  os.path.join(RESULT_DIR, 'rf_model.pkl'))
joblib.dump(xgb_model, os.path.join(RESULT_DIR, 'xgb_model.pkl'))
pd.DataFrame([rf_best_params]).to_csv(
    os.path.join(RESULT_DIR, 'rf_best_params.csv'), index=False)
pd.DataFrame([xgb_best_params]).to_csv(
    os.path.join(RESULT_DIR, 'xgb_best_params.csv'), index=False)
pd.DataFrame({'feature': FEATURES}).to_csv(
    os.path.join(RESULT_DIR, 'features.csv'), index=False)

meta = {
    'gpu_rf': USE_GPU_RF, 'gpu_xgb': USE_GPU_XGB,
    'n_features': len(FEATURES), 'train_records': len(X_train),
    'rf_cv_rmse': rf_cv_rmse,    'xgb_cv_rmse': xgb_cv_rmse,
    'rf_train_rmse': rf_train_m['RMSE'], 'xgb_train_rmse': xgb_train_m['RMSE'],
    'rf_train_r2':   rf_train_m['R2'],   'xgb_train_r2':   xgb_train_m['R2'],
}
if has_val:
    meta.update({
        'val_records': len(X_val),
        'rf_val_rmse': rf_val_m['RMSE'],  'xgb_val_rmse': xgb_val_m['RMSE'],
        'rf_val_r2':   rf_val_m['R2'],    'xgb_val_r2':   xgb_val_m['R2'],
    })
pd.DataFrame([meta]).to_csv(os.path.join(RESULT_DIR, 'run_metadata.csv'), index=False)
print("  Saved: rf_model.pkl  |  xgb_model.pkl")
print("  Saved: rf_best_params.csv  |  xgb_best_params.csv")
print("  Saved: features.csv  |  run_metadata.csv\n")

# ── STEP 8 — Metrics ─────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 8 — Metrics summary")
print("=" * 65)
def fmt(v):
    return round(float(v), 4) if isinstance(v, (int, float, np.floating)) else v

rows = [
    {'Model': 'Random Forest', 'Split': 'Train (CV)',
     'RMSE': fmt(rf_cv_rmse),  'MAE': '-', 'R2': '-', 'MAPE': '-'},
    {'Model': 'Random Forest', 'Split': 'Train',
     **{k: fmt(v) for k, v in rf_train_m.items() if k != 'predictions'}},
    {'Model': 'XGBoost', 'Split': 'Train (CV)',
     'RMSE': fmt(xgb_cv_rmse), 'MAE': '-', 'R2': '-', 'MAPE': '-'},
    {'Model': 'XGBoost', 'Split': 'Train',
     **{k: fmt(v) for k, v in xgb_train_m.items() if k != 'predictions'}},
]
if has_val:
    rows += [
        {'Model': 'Random Forest', 'Split': 'Validate',
         **{k: fmt(v) for k, v in rf_val_m.items() if k != 'predictions'}},
        {'Model': 'XGBoost', 'Split': 'Validate',
         **{k: fmt(v) for k, v in xgb_val_m.items() if k != 'predictions'}},
    ]
metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))
metrics_df.to_csv(os.path.join(RESULT_DIR, 'metrics_summary.csv'), index=False)
print()

# ── STEP 9 — Plots ───────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 9 — Generating plots")
print("=" * 65)
PALETTE = {'Random Forest': '#27ae60', 'XGBoost': '#2980b9'}

# Feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(FEATURES) * 0.55 + 2)))
fig.suptitle('Feature Importance', fontsize=14, fontweight='bold')
for ax, (name, model) in zip(axes, [('Random Forest', rf_model),
                                     ('XGBoost',       xgb_model)]):
    imp  = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
    norm = imp / imp.sum()
    colors = [PALETTE[name] if v >= imp.median() else '#bdc3c7' for v in imp]
    bars = ax.barh(imp.index, norm.values, color=colors, edgecolor='white', height=0.6)
    for bar, val in zip(bars, norm.values):
        ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=8)
    ax.axvline(1 / len(FEATURES), color='red', linestyle='--',
               linewidth=1, label='Equal importance')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Normalised Importance')
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'feature_importance.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: feature_importance.png")

# Feature importance comparison
fig, ax = plt.subplots(figsize=(10, 5))
imp_both = pd.DataFrame({
    'Random Forest': pd.Series(rf_model.feature_importances_,  index=FEATURES) /
                     rf_model.feature_importances_.sum(),
    'XGBoost':       pd.Series(xgb_model.feature_importances_, index=FEATURES) /
                     xgb_model.feature_importances_.sum(),
})
imp_both['mean'] = imp_both.mean(axis=1)
imp_both.sort_values('mean', ascending=False)[['Random Forest','XGBoost']].plot(
    kind='bar', ax=ax,
    color=[PALETTE['Random Forest'], PALETTE['XGBoost']], edgecolor='white')
ax.set_title('Feature Importance — RF vs XGBoost', fontsize=12, fontweight='bold')
ax.set_ylabel('Normalised Importance')
ax.tick_params(axis='x', rotation=30)
ax.legend(); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'feature_importance_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: feature_importance_comparison.png")

# Actual vs Predicted
def plot_avp(y_true, pairs, title, filename):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for ax, (name, m) in zip(axes, pairs):
        preds = m['predictions']
        lim   = [min(y_true.min(), preds.min()) * 0.92,
                 max(y_true.max(), preds.max()) * 1.08]
        ax.scatter(y_true, preds, alpha=0.45, color=PALETTE[name],
                   edgecolors='white', s=40)
        ax.plot(lim, lim, 'k--', linewidth=1.2, label='Perfect fit')
        ax.fill_between(lim, [v * 0.9 for v in lim], [v * 1.1 for v in lim],
                        alpha=0.07, color=PALETTE[name], label='±10% band')
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel('Actual Yield (kg/ha)')
        ax.set_ylabel('Predicted Yield (kg/ha)')
        ax.set_title(f"{name}\nRMSE={m['RMSE']:.2f}  R²={m['R2']:.4f}  "
                     f"MAPE={m['MAPE']:.2f}%", fontsize=10)
        ax.legend(fontsize=8); ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, filename), dpi=150, bbox_inches='tight')
    plt.close()

plot_avp(y_train, [('Random Forest', rf_train_m), ('XGBoost', xgb_train_m)],
         'Actual vs Predicted — Training Set', 'actual_vs_predicted_train.png')
print("  Saved: actual_vs_predicted_train.png")
if has_val:
    plot_avp(y_val, [('Random Forest', rf_val_m), ('XGBoost', xgb_val_m)],
             'Actual vs Predicted — Validation Set (2025)',
             'actual_vs_predicted_validate.png')
    print("  Saved: actual_vs_predicted_validate.png")

# Residuals
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Residuals Analysis — Training Set', fontsize=13, fontweight='bold')
for col, (name, m) in enumerate([('Random Forest', rf_train_m),
                                   ('XGBoost',       xgb_train_m)]):
    res = y_train - m['predictions']
    axes[0, col].scatter(m['predictions'], res, alpha=0.4,
                         color=PALETTE[name], edgecolors='white', s=30)
    axes[0, col].axhline(0, color='red', linestyle='--', linewidth=1.2)
    axes[0, col].set_title(f'{name} — Residuals vs Predicted')
    axes[0, col].set_xlabel('Predicted (kg/ha)')
    axes[0, col].set_ylabel('Residual (kg/ha)')
    axes[0, col].spines[['top', 'right']].set_visible(False)
    axes[1, col].hist(res, bins=35, color=PALETTE[name], edgecolor='white', alpha=0.85)
    axes[1, col].axvline(0, color='red', linestyle='--', linewidth=1.2)
    axes[1, col].axvline(res.mean(), color='orange', linewidth=1.2,
                         label=f'Mean = {res.mean():.1f}')
    axes[1, col].set_title(f'{name} — Residuals Distribution')
    axes[1, col].set_xlabel('Residual (kg/ha)')
    axes[1, col].set_ylabel('Frequency')
    axes[1, col].legend(fontsize=8)
    axes[1, col].spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'residuals_analysis.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: residuals_analysis.png")

# Search history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hyperparameter Search — CV RMSE per Iteration', fontsize=12, fontweight='bold')
for ax, (name, fname) in zip(axes, [('Random Forest', 'random_forest_cv_results.csv'),
                                     ('XGBoost',       'xgboost_cv_results.csv')]):
    cv_df     = pd.read_csv(os.path.join(RESULT_DIR, fname))
    rmse_vals = -cv_df['mean_test_score']
    ax.plot(range(1, len(rmse_vals)+1), rmse_vals,
            alpha=0.35, color=PALETTE[name], linewidth=0.8, label='Trial')
    ax.plot(range(1, len(rmse_vals)+1), rmse_vals.cummin(),
            color=PALETTE[name], linewidth=2, label='Best so far')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Iteration'); ax.set_ylabel('CV RMSE (kg/ha)')
    ax.legend(fontsize=9); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'hyperparam_search_history.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: hyperparam_search_history.png")

# Model comparison
splits   = ['Train (CV)', 'Train'] + (['Validate'] if has_val else [])
rf_rmse_ = [rf_cv_rmse,  rf_train_m['RMSE']]  + ([rf_val_m['RMSE']]  if has_val else [])
xg_rmse_ = [xgb_cv_rmse, xgb_train_m['RMSE']] + ([xgb_val_m['RMSE']] if has_val else [])
rf_r2_   = [np.nan,      rf_train_m['R2']]    + ([rf_val_m['R2']]    if has_val else [])
xg_r2_   = [np.nan,      xgb_train_m['R2']]   + ([xgb_val_m['R2']]   if has_val else [])

x = np.arange(len(splits)); w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model Performance Comparison', fontsize=13, fontweight='bold')
axes[0].bar(x-w/2, rf_rmse_, w, label='Random Forest',
            color=PALETTE['Random Forest'], edgecolor='white')
axes[0].bar(x+w/2, xg_rmse_, w, label='XGBoost',
            color=PALETTE['XGBoost'], edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(splits)
axes[0].set_title('RMSE — lower is better'); axes[0].set_ylabel('RMSE (kg/ha)')
axes[0].legend(); axes[0].spines[['top','right']].set_visible(False)

r2_rf = [v if not np.isnan(v) else 0 for v in rf_r2_]
r2_xg = [v if not np.isnan(v) else 0 for v in xg_r2_]
axes[1].bar(x-w/2, r2_rf, w, label='Random Forest',
            color=PALETTE['Random Forest'], edgecolor='white')
axes[1].bar(x+w/2, r2_xg, w, label='XGBoost',
            color=PALETTE['XGBoost'], edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(splits)
axes[1].set_title('R² — higher is better'); axes[1].set_ylabel('R²')
axes[1].set_ylim(min(0, min(r2_rf+r2_xg)) - 0.05, 1)
axes[1].legend(); axes[1].spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'model_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: model_comparison.png\n")

# ── DONE ─────────────────────────────────────────────────────────────────────
print("=" * 65)
print(f"DONE — All outputs in '{RESULT_DIR}/'")
print("=" * 65)
print(f"""
  Models   : rf_model.pkl  |  xgb_model.pkl
  Params   : rf_best_params.csv  |  xgb_best_params.csv
  CV logs  : random_forest_cv_results.csv  |  xgboost_cv_results.csv
  Metrics  : metrics_summary.csv  |  run_metadata.csv
  Features : features.csv
  Plots    : feature_importance.png
             feature_importance_comparison.png
             actual_vs_predicted_train.png
             {'actual_vs_predicted_validate.png' if has_val else '(validate — pending 2025 data)'}
             residuals_analysis.png
             hyperparam_search_history.png
             model_comparison.png
""")

print("""
─────────────────────────────────────────────────────────────────
INFERENCE EXAMPLE
─────────────────────────────────────────────────────────────────
import joblib, pandas as pd

rf_model  = joblib.load('result/rf_model.pkl')
xgb_model = joblib.load('result/xgb_model.pkl')
features  = pd.read_csv('result/features.csv')['feature'].tolist()

climate_input = pd.DataFrame([{
    'rainfall_mm'               : 1850.0,
    'temperature_celsius'       : 20.3,
    'relative_humidity_percent' : 83.5,
    'soil_moisture_percent'     : 44.0,
    'wind_speed_10m'            : 1.1,
    'dtr_celsius'               : 9.2,
    'vpd_kpa'                   : 0.41,
    'net_solar_rad_kwh_m2'      : 4.6,
}])

luas_lahan_ha = 150.0

for name, model in [('Random Forest', rf_model), ('XGBoost', xgb_model)]:
    yield_kgha = model.predict(climate_input[features])[0]
    total_kg   = yield_kgha * luas_lahan_ha
    print(f"{name}")
    print(f"  Yield            : {yield_kgha:.2f} kg/ha")
    print(f"  Total production : {total_kg:.2f} kg  ({total_kg/1000:.3f} ton)")
─────────────────────────────────────────────────────────────────
""")

[CPU] sklearn RandomForestRegressor  ✓
[GPU] XGBoost (CUDA)  ✓

STEP 1 — Loading production data
  Production years : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Villages         : 218
  Total records    : 1095
  yield_kg_ha      : min=0.75  mean=713.82  max=1166.67

STEP 2 — Loading & aggregating climate data
  Climate years : [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]
  Locations     : 219
  Features      : ['rainfall_mm', 'temperature_celsius', 'relative_humidity_percent', 'soil_moisture_percent', 'wind_speed_10m', 'dtr_celsius', 'vpd_kpa', 'net_solar_rad_kwh_m2']

STEP 3 — Lag merge  (climate year N → production year N+1)
  Merged records   : 1095
  Villages matched : 218
  Production years : [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

STEP 4 — Train / Validate split
  Training   : 876 records  | prod years [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024

In [4]:
import os
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint, uniform, loguniform

warnings.filterwarnings('ignore')

# ── GPU / CPU AUTO-DETECT ─────────────────────────────────────────────────────
USE_GPU_XGB = False
try:
    from xgboost import XGBRegressor
    import subprocess, sys
    r = subprocess.run(
        [sys.executable, '-c',
         'import xgboost as x; x.XGBRegressor(device="cuda").fit([[1]],[1])'],
        capture_output=True, timeout=10)
    USE_GPU_XGB = r.returncode == 0
    print(f"[{'GPU' if USE_GPU_XGB else 'CPU'}] XGBoost "
          f"({'CUDA' if USE_GPU_XGB else 'CPU fallback'})  ✓")
except Exception:
    from xgboost import XGBRegressor
    print("[CPU] XGBoost  ✓")
print()

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG  — adjust file paths if needed
# ─────────────────────────────────────────────────────────────────────────────
PRODUCTION_FILE  = '../data/coffeeProduction_benerMeriah.csv'
CLIMATE_FILE     = '../data/climateData_BenerMeriah_2020-01-01_2024-12-31.csv'
RESULT_DIR       = '../result'
RANDOM_STATE     = 42
CV_FOLDS         = 5
N_ITER_SEARCH    = 100
YIELD_MIN_FILTER = 100   # drop rows with yield < 100 kg/ha (abandoned/outlier villages)

os.makedirs(RESULT_DIR, exist_ok=True)

CLIMATE_BASE_VARS = [
    'rainfall_mm', 'temperature_celsius', 'relative_humidity_percent',
    'soil_moisture_percent', 'wind_speed_10m', 'dtr_celsius',
    'vpd_kpa', 'net_solar_rad_kwh_m2',
]

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Load & clean production data
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 1 — Loading production data")
print("=" * 65)

prod_df = pd.read_csv(PRODUCTION_FILE)
prod_df['village']     = prod_df['village'].str.strip().str.lower()
prod_df['tm_ha']       = pd.to_numeric(prod_df['tm_ha'],       errors='coerce')
prod_df['produksi_kg'] = pd.to_numeric(prod_df['produksi_kg'], errors='coerce')
prod_df = prod_df[prod_df['tm_ha'] > 0].dropna(subset=['tm_ha', 'produksi_kg']).copy()
prod_df['yield_kg_ha'] = prod_df['produksi_kg'] / prod_df['tm_ha']
prod_df = prod_df.dropna(subset=['yield_kg_ha']).reset_index(drop=True)

before = len(prod_df)
prod_df = prod_df[prod_df['yield_kg_ha'] >= YIELD_MIN_FILTER].reset_index(drop=True)

print(f"  Outlier rows removed : {before - len(prod_df)}  "
      f"(yield < {YIELD_MIN_FILTER} kg/ha)")
print(f"  Remaining records    : {len(prod_df)}")
print(f"  Production years     : {sorted(prod_df['year'].unique())}")
print(f"  Villages             : {prod_df['village'].nunique()}")
print(f"  yield_kg_ha          : min={prod_df['yield_kg_ha'].min():.1f}  "
      f"mean={prod_df['yield_kg_ha'].mean():.1f}  "
      f"max={prod_df['yield_kg_ha'].max():.1f}\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Build climate features  (annual mean + per-quarter mean + interactions)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 2 — Building climate features  (annual + quarterly + interactions)")
print("=" * 65)

climate_df             = pd.read_csv(CLIMATE_FILE, parse_dates=['date'])
climate_df['year']     = climate_df['date'].dt.year
climate_df['quarter']  = climate_df['date'].dt.quarter
climate_df['location'] = climate_df['location'].str.strip().str.lower()

VARS = [c for c in CLIMATE_BASE_VARS if c in climate_df.columns]

# Annual mean per village
annual = (climate_df
          .groupby(['year', 'location'])[VARS]
          .mean()
          .reset_index())

# Quarterly mean → wide pivot  (Q1..Q4 for every variable)
quarterly = (climate_df
             .groupby(['year', 'location', 'quarter'])[VARS]
             .mean()
             .reset_index())
quarterly_wide = quarterly.pivot_table(
    index=['year', 'location'], columns='quarter', values=VARS)
quarterly_wide.columns = [f'{v}_Q{q}' for v, q in quarterly_wide.columns]
quarterly_wide = quarterly_wide.reset_index()

# Merge annual + quarterly
climate_feat = pd.merge(annual, quarterly_wide, on=['year', 'location'])

# Pairwise interaction terms for key variables
def add_interactions(df):
    df = df.copy()
    key = [k for k in ['temperature_celsius', 'vpd_kpa',
                        'soil_moisture_percent', 'dtr_celsius', 'rainfall_mm']
           if k in df.columns]
    for i in range(len(key)):
        for j in range(i + 1, len(key)):
            df[f'{key[i]}_x_{key[j]}'] = df[key[i]] * df[key[j]]
    return df

climate_feat = add_interactions(climate_feat)
climate_feat['prod_year'] = climate_feat['year'] + 1   # lag: climate N → yield N+1

CLIMATE_FEATURE_COLS = [c for c in climate_feat.columns
                        if c not in ['year', 'location', 'prod_year']]

print(f"  Annual vars         : {len(VARS)}")
print(f"  Quarterly vars      : {len(VARS) * 4}")
print(f"  Interaction terms   : "
      f"{len([c for c in CLIMATE_FEATURE_COLS if '_x_' in c])}")
print(f"  Total climate feats : {len(CLIMATE_FEATURE_COLS)}\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Lag merge  (climate year N → production year N+1)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 3 — Lag merge  (climate year N → production year N+1)")
print("=" * 65)

merged = pd.merge(
    climate_feat,
    prod_df[['year', 'village', 'tm_ha', 'produksi_kg', 'yield_kg_ha']],
    left_on  = ['prod_year', 'location'],
    right_on = ['year',      'village'],
    how      = 'inner'
).drop(columns=['year_y']).rename(columns={'year_x': 'climate_year'})

print(f"  Merged records   : {len(merged)}")
print(f"  Villages matched : {merged['village'].nunique()}")
print(f"  Production years : {sorted(merged['prod_year'].unique())}\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Add previous year yield  (prev_yield_kg_ha)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 4 — Adding previous year yield feature")
print("=" * 65)

# prev_yield for prod_year N = actual yield in year N-1
prev_yield = (prod_df[['year', 'village', 'yield_kg_ha']]
              .copy()
              .rename(columns={'year': 'prod_year',
                               'yield_kg_ha': 'prev_yield_kg_ha'}))
prev_yield['prod_year'] = prev_yield['prod_year'] + 1   # shift: year N → N+1

merged = pd.merge(
    merged,
    prev_yield[['prod_year', 'village', 'prev_yield_kg_ha']],
    on  = ['prod_year', 'village'],
    how = 'left'
)

# Fill any missing prev_yield with village median across years
village_median = merged.groupby('village')['prev_yield_kg_ha'].transform('median')
merged['prev_yield_kg_ha'] = merged['prev_yield_kg_ha'].fillna(village_median)

missing_prev = merged['prev_yield_kg_ha'].isna().sum()
print(f"  Rows with prev_yield filled by village median : "
      f"{(merged['prev_yield_kg_ha'] == village_median).sum()}")
print(f"  Still missing after fill                     : {missing_prev}")
if missing_prev > 0:
    merged['prev_yield_kg_ha'] = merged['prev_yield_kg_ha'].fillna(
        merged['prev_yield_kg_ha'].median())
print(f"  prev_yield_kg_ha range : "
      f"{merged['prev_yield_kg_ha'].min():.1f} – "
      f"{merged['prev_yield_kg_ha'].max():.1f}\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Train / Validate split  +  log-transform target
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 5 — Split + log-transform target")
print("=" * 65)

# Feature list = all climate features + prev_yield
FEATURES = CLIMATE_FEATURE_COLS + ['prev_yield_kg_ha']

train_df    = merged[merged['prod_year'] <= 2024].copy()
validate_df = merged[merged['prod_year'] == 2025].copy()
has_val     = len(validate_df) > 0

# Log-transform: model learns log1p(yield), outputs converted back via expm1
train_df['log_yield'] = np.log1p(train_df['yield_kg_ha'])
if has_val:
    validate_df['log_yield'] = np.log1p(validate_df['yield_kg_ha'])

print(f"  Training   : {len(train_df)} records  "
      f"| prod years {sorted(train_df['prod_year'].unique())}")
if has_val:
    print(f"  Validation : {len(validate_df)} records  | prod year 2025")
else:
    print("  Validation : 2025 data not available — skipped")
print(f"  Total features : {len(FEATURES)}  "
      f"(climate + prev_yield_kg_ha)")
print(f"  Target         : log1p(yield_kg_ha) → expm1() to get kg/ha back\n")

# Fill any NaN with training median
train_median = train_df[FEATURES].median()
X_train      = train_df[FEATURES].fillna(train_median).values
y_train      = train_df['log_yield'].values
y_train_orig = train_df['yield_kg_ha'].values

if has_val:
    X_val       = validate_df[FEATURES].fillna(train_median).values
    y_val       = validate_df['log_yield'].values
    y_val_orig  = validate_df['yield_kg_ha'].values

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Tuning & evaluation helpers
# ─────────────────────────────────────────────────────────────────────────────
def tune_model(model, param_dist, X, y, name):
    print(f"  Tuning {name}  |  {N_ITER_SEARCH} iters  |  {CV_FOLDS}-fold CV ...")
    cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        estimator           = model,
        param_distributions = param_dist,
        n_iter              = N_ITER_SEARCH,
        scoring             = 'neg_root_mean_squared_error',
        cv                  = cv,
        random_state        = RANDOM_STATE,
        n_jobs              = -1,
        verbose             = 0,
        refit               = True,
    )
    search.fit(X, y)
    cv_rmse_log  = -search.best_score_
    train_pred   = np.expm1(search.best_estimator_.predict(X))
    cv_rmse_kgha = np.sqrt(mean_squared_error(np.expm1(y), train_pred))

    print(f"  Best CV RMSE (log space) : {cv_rmse_log:.5f}")
    print(f"  Train RMSE (kg/ha)       : {cv_rmse_kgha:.2f}")
    print(f"  Best params              :")
    for k, v in search.best_params_.items():
        print(f"    {k:32s} = {v}")

    (pd.DataFrame(search.cv_results_)
       .sort_values('rank_test_score')
       .to_csv(os.path.join(RESULT_DIR,
               f'{name.lower().replace(" ","_")}_cv_results.csv'),
               index=False))
    print()
    return search.best_estimator_, search.best_params_, cv_rmse_log


def evaluate(model, X, y_log, y_orig, label):
    """Predict in log space, convert back to kg/ha, report all metrics."""
    pred_log  = model.predict(X)
    pred_kgha = np.expm1(pred_log)
    rmse = np.sqrt(mean_squared_error(y_orig, pred_kgha))
    mae  = mean_absolute_error(y_orig, pred_kgha)
    r2   = r2_score(y_orig, pred_kgha)
    mape = np.mean(np.abs((y_orig - pred_kgha) /
                           np.where(y_orig == 0, 1, y_orig))) * 100
    print(f"    [{label}]  RMSE={rmse:.2f} kg/ha  MAE={mae:.2f}  "
          f"R²={r2:.4f}  MAPE={mape:.2f}%")
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape,
            'pred_log': pred_log, 'pred_kgha': pred_kgha}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — Random Forest
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 7 — Random Forest  [CPU — sklearn]")
print("=" * 65)

rf_param_dist = {
    'n_estimators'             : randint(100, 800),
    'max_depth'                : [None, 5, 8, 10, 15, 20, 30],
    'min_samples_split'        : randint(2, 20),
    'min_samples_leaf'         : randint(1, 15),
    'max_features'             : ['sqrt', 'log2', 0.3, 0.5, 0.7, 0.8],
    'bootstrap'                : [True, False],
    'max_leaf_nodes'           : [None, 50, 100, 200, 500],
    'min_impurity_decrease'    : uniform(0.0, 0.02),
}

rf_model, rf_best_params, rf_cv = tune_model(
    RandomForestRegressor(random_state=RANDOM_STATE),
    rf_param_dist, X_train, y_train, "Random Forest")

print("  Performance (kg/ha):")
rf_train_m = evaluate(rf_model, X_train, y_train, y_train_orig, "Train")
if has_val:
    rf_val_m = evaluate(rf_model, X_val, y_val, y_val_orig, "Validate")
print()

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8 — XGBoost
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print(f"STEP 8 — XGBoost  [{'GPU — CUDA' if USE_GPU_XGB else 'CPU'}]")
print("=" * 65)

xgb_param_dist = {
    'n_estimators'      : randint(100, 800),
    'learning_rate'     : loguniform(1e-3, 0.3),
    'max_depth'         : randint(2, 10),
    'min_child_weight'  : randint(1, 15),
    'subsample'         : uniform(0.5, 0.5),
    'colsample_bytree'  : uniform(0.5, 0.5),
    'colsample_bylevel' : uniform(0.5, 0.5),
    'gamma'             : uniform(0.0, 1.0),
    'reg_alpha'         : loguniform(1e-4, 5.0),
    'reg_lambda'        : loguniform(1e-3, 5.0),
    'max_bin'           : [128, 256, 512],
    'grow_policy'       : ['depthwise', 'lossguide'],
    'max_leaves'        : randint(0, 32),
}

xgb_model, xgb_best_params, xgb_cv = tune_model(
    XGBRegressor(random_state=RANDOM_STATE, verbosity=0,
                 objective='reg:squarederror', tree_method='hist',
                 device='cuda' if USE_GPU_XGB else 'cpu'),
    xgb_param_dist, X_train, y_train, "XGBoost")

print("  Performance (kg/ha):")
xgb_train_m = evaluate(xgb_model, X_train, y_train, y_train_orig, "Train")
if has_val:
    xgb_val_m = evaluate(xgb_model, X_val, y_val, y_val_orig, "Validate")
print()

# ─────────────────────────────────────────────────────────────────────────────
# STEP 9 — Save models
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 9 — Saving models")
print("=" * 65)

joblib.dump(rf_model,  os.path.join(RESULT_DIR, 'rf_model.pkl'))
joblib.dump(xgb_model, os.path.join(RESULT_DIR, 'xgb_model.pkl'))

pd.DataFrame([rf_best_params]).to_csv(
    os.path.join(RESULT_DIR, 'rf_best_params.csv'), index=False)
pd.DataFrame([xgb_best_params]).to_csv(
    os.path.join(RESULT_DIR, 'xgb_best_params.csv'), index=False)
pd.DataFrame({'feature': FEATURES}).to_csv(
    os.path.join(RESULT_DIR, 'features.csv'), index=False)
pd.DataFrame([{
    'note': 'target is log1p(yield_kg_ha); use np.expm1(model.predict(...)) to get kg/ha'
}]).to_csv(os.path.join(RESULT_DIR, 'model_notes.csv'), index=False)

# Select & save best model
labels    = ['Random Forest', 'XGBoost']
models    = [rf_model, xgb_model]
all_train = [rf_train_m, xgb_train_m]
all_val   = [rf_val_m, xgb_val_m] if has_val else []

if has_val:
    best_idx = np.argmin([m['RMSE'] for m in all_val])
else:
    best_idx = np.argmin([m['RMSE'] for m in all_train])
best_name  = labels[best_idx]
best_model = models[best_idx]
joblib.dump(best_model, os.path.join(RESULT_DIR, 'best_model.pkl'))
pd.DataFrame([{'best_model': best_name}]).to_csv(
    os.path.join(RESULT_DIR, 'best_model_name.csv'), index=False)

# Save training median for inference imputation
train_median.to_csv(os.path.join(RESULT_DIR, 'train_feature_median.csv'))

print("  Saved: rf_model.pkl  |  xgb_model.pkl  |  best_model.pkl")
print("  Saved: rf_best_params.csv  |  xgb_best_params.csv")
print("  Saved: features.csv  |  model_notes.csv")
print("  Saved: train_feature_median.csv")
print(f"  Best model → {best_name}\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 10 — Metrics summary
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 10 — Metrics summary")
print("=" * 65)

def fmt(v):
    return round(float(v), 4) if isinstance(v, (int, float, np.floating)) else v

rows = []
for lbl, tm in zip(labels, all_train):
    rows.append({'Model': lbl, 'Split': 'Train',
                 **{k: fmt(v) for k, v in tm.items()
                    if k not in ('pred_log', 'pred_kgha')}})
if has_val:
    for lbl, vm in zip(labels, all_val):
        rows.append({'Model': lbl, 'Split': 'Validate',
                     **{k: fmt(v) for k, v in vm.items()
                        if k not in ('pred_log', 'pred_kgha')}})

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))
metrics_df.to_csv(os.path.join(RESULT_DIR, 'metrics_summary.csv'), index=False)
print(f"\n  → Best model by {'Validate' if has_val else 'Train'} RMSE : {best_name}\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 11 — Plots
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 11 — Generating plots")
print("=" * 65)

COLORS = {'Random Forest': '#27ae60', 'XGBoost': '#2980b9'}

# ── 11a. Actual vs Predicted ──────────────────────────────────────────────────
def plot_avp(y_orig, pairs, title, filename):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for ax, (name, m) in zip(axes, pairs):
        p   = m['pred_kgha']
        lim = [min(y_orig.min(), p.min()) * 0.92,
               max(y_orig.max(), p.max()) * 1.08]
        ax.scatter(y_orig, p, alpha=0.45, color=COLORS[name],
                   edgecolors='white', s=35)
        ax.plot(lim, lim, 'k--', lw=1.3, label='Perfect fit')
        ax.fill_between(lim, [v * 0.9 for v in lim], [v * 1.1 for v in lim],
                        alpha=0.07, color=COLORS[name], label='±10% band')
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel('Actual Yield (kg/ha)', fontsize=10)
        ax.set_ylabel('Predicted Yield (kg/ha)', fontsize=10)
        ax.set_title(
            f"{name}\n"
            f"RMSE={m['RMSE']:.2f} kg/ha  |  MAE={m['MAE']:.2f}\n"
            f"R²={m['R2']:.4f}  |  MAPE={m['MAPE']:.2f}%",
            fontsize=9)
        ax.legend(fontsize=8)
        ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, filename), dpi=150, bbox_inches='tight')
    plt.close()

plot_avp(y_train_orig,
         [('Random Forest', rf_train_m), ('XGBoost', xgb_train_m)],
         'Actual vs Predicted — Training Set',
         'actual_vs_predicted_train.png')
print("  Saved: actual_vs_predicted_train.png")

if has_val:
    plot_avp(y_val_orig,
             [('Random Forest', rf_val_m), ('XGBoost', xgb_val_m)],
             'Actual vs Predicted — Validation Set (2025)',
             'actual_vs_predicted_validate.png')
    print("  Saved: actual_vs_predicted_validate.png")

# ── 11b. Model comparison (RMSE / R² / MAPE) ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Performance Comparison', fontsize=13, fontweight='bold')

x = np.arange(len(labels))
w = 0.35

for ax, (metric, title_m, lower) in zip(axes, [
        ('RMSE', 'RMSE (kg/ha)\nlower = better',  True),
        ('R2',   'R²\nhigher = better',            False),
        ('MAPE', 'MAPE (%)\nlower = better',       True)]):

    train_vals = [m[metric] for m in all_train]
    bars1 = ax.bar(x - (w/2 if has_val else 0), train_vals, w,
                   label='Train',
                   color=[COLORS[l] for l in labels],
                   edgecolor='white', alpha=0.95)
    # value labels on bars
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + ax.get_ylim()[1]*0.01,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=7)

    if has_val:
        val_vals = [m[metric] for m in all_val]
        bars2 = ax.bar(x + w/2, val_vals, w,
                       label='Validate',
                       color=[COLORS[l] for l in labels],
                       edgecolor='white', alpha=0.5)
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + ax.get_ylim()[1]*0.01,
                    f'{bar.get_height():.2f}',
                    ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_title(title_m, fontsize=10)
    if has_val:
        ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'model_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: model_comparison.png")

# ── 11c. Feature importance (top 20) ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, max(7, min(len(FEATURES), 20) * 0.45 + 2)))
fig.suptitle('Feature Importance — Top 20', fontsize=14, fontweight='bold')

for ax, (name, model) in zip(axes, [('Random Forest', rf_model),
                                     ('XGBoost',       xgb_model)]):
    imp  = pd.Series(model.feature_importances_, index=FEATURES)
    imp  = (imp / imp.sum()).sort_values(ascending=False).head(20)
    imp  = imp.sort_values(ascending=True)   # horizontal bar order
    bar_colors = [COLORS[name] if v >= imp.median() else '#bdc3c7' for v in imp]
    bars = ax.barh(imp.index, imp.values,
                   color=bar_colors, edgecolor='white', height=0.6)
    ax.axvline(1 / len(FEATURES), color='red', linestyle='--',
               lw=1, label='Equal share')
    for bar, val in zip(bars, imp.values):
        ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=7)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Normalised Importance')
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'feature_importance.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: feature_importance.png")

# ── 11d. Feature importance side-by-side comparison ──────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
imp_rf  = pd.Series(rf_model.feature_importances_,  index=FEATURES)
imp_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURES)
imp_both = pd.DataFrame({
    'Random Forest': imp_rf  / imp_rf.sum(),
    'XGBoost':       imp_xgb / imp_xgb.sum(),
})
imp_both['mean'] = imp_both.mean(axis=1)
top20 = imp_both.sort_values('mean', ascending=False).head(20)
top20[['Random Forest', 'XGBoost']].plot(
    kind='bar', ax=ax,
    color=[COLORS['Random Forest'], COLORS['XGBoost']],
    edgecolor='white')
ax.set_title('Top 20 Features — RF vs XGBoost', fontsize=12, fontweight='bold')
ax.set_ylabel('Normalised Importance')
ax.tick_params(axis='x', rotation=40, labelsize=8)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'feature_importance_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: feature_importance_comparison.png")

# ── 11e. Residuals ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Residuals Analysis — Training Set', fontsize=13, fontweight='bold')

for col, (name, m) in enumerate(zip(labels, all_train)):
    res = y_train_orig - m['pred_kgha']
    # Residuals vs Predicted
    axes[0, col].scatter(m['pred_kgha'], res, alpha=0.35,
                         color=COLORS[name], edgecolors='white', s=25)
    axes[0, col].axhline(0, color='red', linestyle='--', lw=1.2)
    axes[0, col].set_title(f'{name} — Residuals vs Predicted', fontsize=10)
    axes[0, col].set_xlabel('Predicted (kg/ha)')
    axes[0, col].set_ylabel('Residual (kg/ha)')
    axes[0, col].spines[['top', 'right']].set_visible(False)
    # Distribution
    axes[1, col].hist(res, bins=35, color=COLORS[name],
                      edgecolor='white', alpha=0.85)
    axes[1, col].axvline(0, color='red', linestyle='--', lw=1.2)
    axes[1, col].axvline(res.mean(), color='orange', lw=1.5,
                         label=f'Mean = {res.mean():.1f} kg/ha')
    axes[1, col].set_title(f'{name} — Residuals Distribution', fontsize=10)
    axes[1, col].set_xlabel('Residual (kg/ha)')
    axes[1, col].set_ylabel('Frequency')
    axes[1, col].legend(fontsize=8)
    axes[1, col].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'residuals_analysis.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: residuals_analysis.png")

# ── 11f. Hyperparameter search history ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hyperparameter Search — CV RMSE per Iteration',
             fontsize=12, fontweight='bold')

for ax, (name, fname) in zip(axes, [
        ('Random Forest', 'random_forest_cv_results.csv'),
        ('XGBoost',       'xgboost_cv_results.csv')]):
    cv_df     = pd.read_csv(os.path.join(RESULT_DIR, fname))
    rmse_vals = -cv_df['mean_test_score']
    ax.plot(range(1, len(rmse_vals)+1), rmse_vals,
            alpha=0.3, color=COLORS[name], lw=0.8, label='Trial')
    ax.plot(range(1, len(rmse_vals)+1), rmse_vals.cummin(),
            color=COLORS[name], lw=2, label='Best so far')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('CV RMSE (log space)')
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'hyperparam_search_history.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: hyperparam_search_history.png")

# ── 11g. Yield distribution (raw → filtered → log) ───────────────────────────
prod_raw = pd.read_csv(PRODUCTION_FILE)
prod_raw['village']     = prod_raw['village'].str.strip()
prod_raw['tm_ha']       = pd.to_numeric(prod_raw['tm_ha'],       errors='coerce')
prod_raw['produksi_kg'] = pd.to_numeric(prod_raw['produksi_kg'], errors='coerce')
prod_raw = prod_raw[prod_raw['tm_ha'] > 0].dropna(subset=['tm_ha','produksi_kg'])
prod_raw['yield_kg_ha'] = prod_raw['produksi_kg'] / prod_raw['tm_ha']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Target Variable — Transformation Pipeline',
             fontsize=12, fontweight='bold')

axes[0].hist(prod_raw['yield_kg_ha'], bins=50, color='#95a5a6', edgecolor='white')
axes[0].axvline(YIELD_MIN_FILTER, color='red', linestyle='--', lw=1.5,
                label=f'Filter < {YIELD_MIN_FILTER}')
axes[0].set_title('1. Raw yield_kg_ha')
axes[0].set_xlabel('kg/ha')
axes[0].legend(fontsize=9)
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].hist(train_df['yield_kg_ha'], bins=50, color='#27ae60', edgecolor='white')
axes[1].set_title(f'2. After filter  (≥ {YIELD_MIN_FILTER} kg/ha)')
axes[1].set_xlabel('kg/ha')
axes[1].spines[['top', 'right']].set_visible(False)

axes[2].hist(train_df['log_yield'], bins=50, color='#2980b9', edgecolor='white')
axes[2].set_title('3. log1p(yield)  — model target')
axes[2].set_xlabel('log1p(kg/ha)')
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'yield_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: yield_distribution.png\n")

# ─────────────────────────────────────────────────────────────────────────────
# DONE
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print(f"DONE — All outputs saved to '{RESULT_DIR}/'")
print("=" * 65)

[GPU] XGBoost (CUDA)  ✓

STEP 1 — Loading production data
  Outlier rows removed : 0  (yield < 100 kg/ha)
  Remaining records    : 1095
  Production years     : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Villages             : 219
  yield_kg_ha          : min=586.8  mean=787.3  max=987.1

STEP 2 — Building climate features  (annual + quarterly + interactions)
  Annual vars         : 8
  Quarterly vars      : 32
  Interaction terms   : 10
  Total climate feats : 50

STEP 3 — Lag merge  (climate year N → production year N+1)
  Merged records   : 1095
  Villages matched : 219
  Production years : [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

STEP 4 — Adding previous year yield feature
  Rows with prev_yield filled by village median : 219
  Still missing after fill                     : 0
  prev_yield_kg_ha range : 586.8 – 983.2

STEP 5 — Split + log-transform target
  Training   : 876 records  | prod years [np.int